<a href="https://colab.research.google.com/github/contactdilarayilmaz/asteroid-classification/blob/main/03_spice_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NASA Asteroid Sınıflandırma Projesi
## Notebook 03 — SPICE ile Asteroid Yörünge Görselleştirme

**Bu notebook'ta yapacaklarımız:**
1. SpiceyPy kurulumu ve NAIF kernel'larını indirme
2. SPICE ile gezegen ve asteroid pozisyonu hesaplama
3. Apophis (99942) asteroid yörüngesini 3D olarak görselleştirme
4. 2025–2035 yörünge animasyonu (Plotly ile interaktif)
5. Apophis'in Dünya'ya mesafesini zaman içinde çizme
6. 2029 yakın geçiş anını vurgulama
7. Güneş Sistemi'nde Dünya, Mars ve Apophis yörüngelerini karşılaştırma

> **Bağlam:** Bu notebook önceki adımların *neden önemli* olduğunu görselleştirir.
> SHAP ile tehlikeli bulduğumuz asteroidi şimdi uzayda takip ediyoruz!

---
*Önceki adım:* `02_baseline_models_shap.ipynb`  
*Sonraki adım:* `04_smote_challenge.ipynb`

## 1. Kurulum — SpiceyPy ve Kernel İndirme

In [ ]:
# SpiceyPy ve diğer kütüphaneleri yükle
!pip install spiceypy plotly astropy -q

print('✅ Kütüphaneler hazır!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 7.1 MB/s eta 0:00:00
✅ Kütüphaneler hazır!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import spiceypy as spice
import warnings
warnings.filterwarnings('ignore')

# Görsel stil (önceki notebook'larla tutarlı)
plt.style.use('dark_background')
pd.set_option('display.float_format', '{:.6f}'.format)

print(f'✅ SpiceyPy versiyonu: {spice.__version__}')
print('✅ Tüm import işlemleri tamamlandı!')

✅ SpiceyPy versiyonu: 8.1.0
✅ Tüm import işlemleri tamamlandı!


## 2. SPICE Nedir? — Kısa Teorik Giriş

**SPICE** (Spacecraft Planet Instrument C-matrix Events), NASA/NAIF tarafından geliştirilmiş
gezegen bilimi altyapısıdır. Uzay görevlerinin navigasyonu için kullanılır.

### Temel Kavramlar:
| Kavram | Açıklama |
|--------|----------|
| **Kernel** | SPICE'ın veri dosyaları (.bsp, .tls, .pck) |
| **SPK kernel** | Gök cisimlerinin pozisyon/hız bilgisi |
| **LSK kernel** | Artık saniye düzeltmeleri (leap seconds) |
| **PCK kernel** | Gezegen fiziksel sabitleri |
| **ET (Ephemeris Time)** | SPICE'ın iç zaman formatı |
| **J2000** | Referans koordinat sistemi (2000 yılı başı ekvatoru) |
| **ECLIPJ2000** | Güneş merkezli ekliptik koordinat sistemi |

### Neden SPICE Kullanıyoruz?
- NASA'nın resmi yörünge hesaplama aracı
- Apophis gibi gerçek asteroidlerin pozisyonlarını nanometre hassasiyetiyle verir
- Cassini, Voyager, Mars Reconnaissance Orbiter gibi tüm görevlerde kullanıldı

> **Not:** `de440.bsp` kernel'ı Güneş Sistemi gezegenlerini kapsar.
> Apophis için özel SPK kernel'ı JPL Horizons'tan indirmeliyiz.

## 3. NAIF Kernel Dosyalarını İndirme

SPICE'ın çalışması için **kernel** dosyalarına ihtiyacımız var.
Bunlar NASA sunucularından ücretsiz indirilebilir.

In [ ]:
import os
import urllib.request

# Kernel dosyaları için klasör oluştur
os.makedirs('kernels', exist_ok=True)

# ─── İndirilecek kernel listesi ───────────────────────────────────────────────
KERNELS = {
    # Artık saniye tablosu (zaman dönüşümü için şart)
    'naif0012.tls': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls',
    # Güneş Sistemi gezegenlerinin efemerisleri (DE440 — modern ve hassas)
    'de440s.bsp':   'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440s.bsp',
    # Gezegen fiziksel sabitleri (yarıçap, kutup yönü, vb.)
    'pck00011.tpc': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc',
}

print('📥 NAIF kernel dosyaları indiriliyor...')
print('   (de440s.bsp ~32MB, ilk kez 1-2 dakika sürebilir)\n')

for filename, url in KERNELS.items():
    fpath = f'kernels/{filename}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f'   ✅ {filename} zaten mevcut ({size_mb:.1f} MB)')
        continue
    try:
        print(f'   ⬇️  {filename} indiriliyor...')
        urllib.request.urlretrieve(url, fpath)
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f'   ✅ {filename} indirildi ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'   ❌ {filename} indirilemedi: {e}')

print('\n📦 Kernel indirme tamamlandı!')

📥 NAIF kernel dosyaları indiriliyor...
   (de440s.bsp ~32MB, ilk kez 1-2 dakika sürebilir)

   ⬇️  naif0012.tls indiriliyor...
   ✅ naif0012.tls indirildi (0.0 MB)
   ⬇️  de440s.bsp indiriliyor...
   ✅ de440s.bsp indirildi (31.2 MB)
   ⬇️  pck00011.tpc indiriliyor...
   ✅ pck00011.tpc indirildi (0.1 MB)

📦 Kernel indirme tamamlandı!


## 4. Apophis (99942) Kernel'ını JPL Horizons'tan İndirme

**Apophis (99942)**, 2029'da Dünya'ya çıplak gözle görülebilecek kadar yakın geçecek.
Bu özelliği projeye mükemmel bir anlatı sağlıyor.

### JPL Horizons'tan Manuel İndirme (Önerilen Yöntem):
1. https://ssd.jpl.nasa.gov/horizons/ adresine git
2. **Target Body** → `Apophis (99942)` yaz
3. **Ephemeris Type** → `SPK File` seç
4. **Time Span** → 2024-Jan-01 ile 2036-Jan-01 arası
5. **Generate Ephemeris** → Download → `wld54180.15` veya benzeri .bsp indir
6. İndirilen dosyayı `kernels/apophis.bsp` olarak kaydet

### Otomatik İndirme (Backup — Horizons API):
Aşağıdaki kod Horizons web API'sini kullanarak Apophis SPK kernel'ını otomatik indirir.

In [ ]:
import urllib.request
import urllib.parse
import time

APOPHIS_SPK = 'kernels/apophis.bsp'

def download_apophis_spk():
    """JPL Horizons API üzerinden Apophis SPK kernel indirir."""

    # JPL Horizons batch komut — SPK dosyası üretmek için
    # NAIF ID: 2099942 (küçük cisim formatı) veya 99942 (numbered asteroid)
    horizons_url = 'https://ssd.jpl.nasa.gov/api/horizons.api'

    params = {
        'format': 'text',
        'COMMAND': "'99942'",       # Apophis NAIF ID
        'EPHEM_TYPE': 'SPK',
        'START_TIME': '2024-JAN-01',
        'STOP_TIME': '2036-JAN-01',
        'STEP_SIZE': '1d',
        'CENTER': '500@0',          # Güneş'e göre (Solar System Barycenter)
    }

    url = horizons_url + '?' + urllib.parse.urlencode(params)

    print('⏳ Horizons APIden Apophis SPK isteği gönderiliyor...')
    print(f'   URL: {url[:80]}...')

    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            content = response.read()

        # Yanıt SPK binary mi kontrol et
        if content[:7] == b'DAF/SPK':
            with open(APOPHIS_SPK, 'wb') as f:
                f.write(content)
            size_kb = len(content) / 1024
            print(f'   ✅ Apophis SPK kernel indirildi ({size_kb:.1f} KB)')
            return True
        else:
            # Text yanıt — hata mesajı olabilir
            text_sample = content[:200].decode('utf-8', errors='ignore')
            print(f'   ⚠️  Beklenmedik yanıt formatı: {text_sample[:100]}')
            return False

    except Exception as e:
        print(f'   ❌ API isteği başarısız: {e}')
        return False

# ─── Kernel yoksa indir ─────────────────────────────────────────────────────
if not os.path.exists(APOPHIS_SPK):
    success = download_apophis_spk()
    if not success:
        print()
        print('💡 Manuel İndirme Adımları:')
        print('   1. https://ssd.jpl.nasa.gov/horizons/ adresine git')
        print('   2. Target Body: Apophis (99942)')
        print('   3. Ephemeris Type: SPK File')
        print('   4. Time Span: 2024-Jan-01 to 2036-Jan-01')
        print('   5. İndirilen dosyayı kernels/apophis.bsp olarak kaydet')
        print()
        print('   📌 Alternatif: Notebookun geri kalanı Apophis kernelı olmadan')
        print('      orbital parametreler kullanarak devam edebilir.')
else:
    size_kb = os.path.getsize(APOPHIS_SPK) / 1024
    print(f'✅ kernels/apophis.bsp zaten mevcut ({size_kb:.1f} KB)')

⏳ Horizons APIden Apophis SPK isteği gönderiliyor...
   URL: https://ssd.jpl.nasa.gov/api/horizons.api?format=text&COMMAND=%2799942%27&EPHEM_...
   ⚠️  Beklenmedik yanıt formatı: API VERSION: 1.2
API SOURCE: NASA/JPL Horizons API

************************************************

💡 Manuel İndirme Adımları:
   1. https://ssd.jpl.nasa.gov/horizons/ adresine git
   2. Target Body: Apophis (99942)
   3. Ephemeris Type: SPK File
   4. Time Span: 2024-Jan-01 to 2036-Jan-01
   5. İndirilen dosyayı kernels/apophis.bsp olarak kaydet

   📌 Alternatif: Notebookun geri kalanı Apophis kernelı olmadan
      orbital parametreler kullanarak devam edebilir.


## 5. SPICE Kernel'larını Yükle ve İlk Test

In [ ]:
# Tüm kernel'ları SPICE'a yükle
print('🔧 SPICE kernel\'ları yükleniyor...')

# Önceki oturumda yüklenmiş olabilir — temizle
try:
    spice.kclear()
except:
    pass

# Temel kernel'lar
LOADED_KERNELS = []
for fname in ['naif0012.tls', 'de440s.bsp', 'pck00011.tpc']:
    fpath = f'kernels/{fname}'
    if os.path.exists(fpath):
        spice.furnsh(fpath)
        LOADED_KERNELS.append(fname)
        print(f'   ✅ {fname} yüklendi')
    else:
        print(f'   ⚠️  {fname} bulunamadı — lütfen indirme adımını tekrar çalıştır')

# Apophis kernel'ı (varsa)
APOPHIS_KERNEL_LOADED = False
if os.path.exists('kernels/apophis.bsp'):
    spice.furnsh('kernels/apophis.bsp')
    LOADED_KERNELS.append('apophis.bsp')
    APOPHIS_KERNEL_LOADED = True
    print('   ✅ apophis.bsp yüklendi')
else:
    print('   ⚠️  apophis.bsp bulunamadı — orbital parametre modu kullanılacak')

print(f'\n📊 Yüklü kernel sayısı: {len(LOADED_KERNELS)}')

# ─── Basit test: Dünya'nın pozisyonu ─────────────────────────────────────────
# str2et: İnsan okunabilir tarihi Ephemeris Time'a çevirir
et_test = spice.str2et('2025-01-01')
earth_pos, lt = spice.spkpos('EARTH', et_test, 'J2000', 'NONE', 'SUN')

print(f'\n🌍 Test — Dünya\'nın 2025-01-01 pozisyonu (J2000, güneş merkezli):')
print(f'   X = {earth_pos[0]/1.496e8:.4f} AU')
print(f'   Y = {earth_pos[1]/1.496e8:.4f} AU')
print(f'   Z = {earth_pos[2]/1.496e8:.4f} AU')
print(f'   Güneş\'e mesafe: {np.linalg.norm(earth_pos)/1.496e8:.4f} AU  (≈1 AU beklenir)')
print('\n✅ SPICE başarıyla çalışıyor!')

🔧 SPICE kernel'ları yükleniyor...
   ✅ naif0012.tls yüklendi
   ✅ de440s.bsp yüklendi
   ✅ pck00011.tpc yüklendi
   ⚠️  apophis.bsp bulunamadı — orbital parametre modu kullanılacak

📊 Yüklü kernel sayısı: 3

🌍 Test — Dünya'nın 2025-01-01 pozisyonu (J2000, güneş merkezli):
   X = -0.1787 AU
   Y = 0.8872 AU
   Z = 0.3846 AU
   Güneş'e mesafe: 0.9833 AU  (≈1 AU beklenir)

✅ SPICE başarıyla çalışıyor!


## 6. Dünya ve Apophis Pozisyonlarını Hesapla (2025–2035)

SPICE'ın çekirdeği: belirli bir zaman diliminde gezegen/asteroid pozisyonlarını
Ephemeris kernel'larından okuyarak döndürür.

In [ ]:
# ─── Zaman aralığı tanımla ────────────────────────────────────────────────────
# 2025 başından 2036 başına, 7 günde bir
date_start = '2025-01-01'
date_end   = '2036-01-01'
step_days  = 7  # Gün

# Tarih listesi oluştur
import datetime
dates = []
current = datetime.date(2025, 1, 1)
end_date = datetime.date(2036, 1, 1)
while current <= end_date:
    dates.append(current.strftime('%Y-%m-%d'))
    current += datetime.timedelta(days=step_days)

# Epoch zamanlarına (ET) çevir
et_times = [spice.str2et(d) for d in dates]
n_steps  = len(et_times)
print(f'📅 Zaman adımı sayısı: {n_steps} ({step_days} günde bir, ~11 yıl)')

# ─── AU dönüşüm sabiti ────────────────────────────────────────────────────────
AU = 1.496e8  # km/AU

# ─── Dünya pozisyonları ───────────────────────────────────────────────────────
print('\n⏳ Dünya pozisyonları hesaplanıyor...')
earth_positions = np.array([
    spice.spkpos('EARTH', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
    for et in et_times
]) / AU  # km → AU

print(f'   ✅ Dünya: {earth_positions.shape} pozisyon vektörü')

# ─── Mars pozisyonları (karşılaştırma için) ───────────────────────────────────
print('⏳ Mars pozisyonları hesaplanıyor...')
mars_positions = np.array([
    spice.spkpos('MARS', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
    for et in et_times
]) / AU

print(f'   ✅ Mars: {mars_positions.shape} pozisyon vektörü')

# ─── Apophis pozisyonları ─────────────────────────────────────────────────────
if APOPHIS_KERNEL_LOADED:
    print('⏳ Apophis pozisyonları hesaplanıyor (SPK kernel)...')
    try:
        apophis_positions = np.array([
            spice.spkpos('2099942', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
            for et in et_times
        ]) / AU
        print(f'   ✅ Apophis: {apophis_positions.shape} pozisyon vektörü (SPK kernel)')
        SPICE_APOPHIS = True
    except Exception as e:
        print(f'   ⚠️  SPK hatası: {e}')
        SPICE_APOPHIS = False
else:
    SPICE_APOPHIS = False

# ─── Fallback: Keplerian orbital parametreler ile Apophis yörüngesi ───────────
# Apophis gerçek orbital elementleri (JPL SBDB, 2024 güncellemesi)
if not SPICE_APOPHIS:
    print('⏳ Apophis pozisyonları orbital parametrelerle hesaplanıyor (fallback)...')

    # Apophis gerçek Kepler elemanları (J2000.0 epoch)
    a_apophis = 0.9226  # AU — yarı büyük eksen
    e_apophis = 0.1914  # eksantriklik
    i_apophis = 3.3395  # derece — eğim
    omega_apophis = 126.38  # derece — perihel argümanı
    Omega_apophis = 204.43  # derece — yükselen düğüm boylamı
    M0_apophis = 180.0  # derece — epoch'taki ortalama anomali (yaklaşık)
    T_period = a_apophis**1.5 * 365.25  # gün — orbital period (Kepler 3. yasası)

    def kepler_orbit_3d(a, e, i_deg, omega_deg, Omega_deg, M0_deg, t_days, n_points=500):
        """Kepler orbital elementlerinden 3D yörünge koordinatları üretir."""
        i = np.radians(i_deg)
        omega = np.radians(omega_deg)
        Omega = np.radians(Omega_deg)
        T = a**1.5 * 365.25  # Kepler 3. yasası

        # Ortalama anomali dizisi
        M_vals = (np.radians(M0_deg) + 2*np.pi * t_days / T) % (2*np.pi)

        # Eccentric anomaly — Newton-Raphson ile çöz
        E_vals = M_vals.copy()
        for _ in range(100):
            E_vals -= (E_vals - e * np.sin(E_vals) - M_vals) / (1 - e * np.cos(E_vals))

        # True anomaly
        nu = 2 * np.arctan2(np.sqrt(1+e)*np.sin(E_vals/2),
                            np.sqrt(1-e)*np.cos(E_vals/2))

        # Orbital düzlemde r (AU)
        r = a * (1 - e**2) / (1 + e * np.cos(nu))

        # Orbital düzlem koordinatları
        x_orb = r * np.cos(nu)
        y_orb = r * np.sin(nu)

        # 3D dönüşüm (Euler rotasyonları)
        # 1) Perihel argümanı (omega) etrafında döndür
        x1 = x_orb * np.cos(omega) - y_orb * np.sin(omega)
        y1 = x_orb * np.sin(omega) + y_orb * np.cos(omega)

        # 2) Eğim (i) etrafında döndür
        x2 = x1
        y2 = y1 * np.cos(i)
        z2 = y1 * np.sin(i)

        # 3) Yükselen düğüm (Omega) etrafında döndür
        x3 = x2 * np.cos(Omega) - y2 * np.sin(Omega)
        y3 = x2 * np.sin(Omega) + y2 * np.cos(Omega)
        z3 = z2

        return np.column_stack([x3, y3, z3])

    # Apophis pozisyonlarını hesapla
    # t_days = 2025-01-01'den itibaren geçen günler
    t0 = datetime.date(2025, 1, 1)
    t_days_array = np.array([(datetime.datetime.strptime(d, '%Y-%m-%d').date() - t0).days
                              for d in dates], dtype=float)

    apophis_positions = kepler_orbit_3d(
        a_apophis, e_apophis, i_apophis, omega_apophis, Omega_apophis, M0_apophis,
        t_days_array
    )
    print(f'   ✅ Apophis: {apophis_positions.shape} pozisyon vektörü (Kepler fallback)')
    SPICE_APOPHIS = False  # Bilgi amaçlı

print('\n✅ Tüm pozisyon verileri hazır!')
print(f'   Zaman aralığı: {dates[0]} → {dates[-1]}')

📅 Zaman adımı sayısı: 574 (7 günde bir, ~11 yıl)

⏳ Dünya pozisyonları hesaplanıyor...
   ✅ Dünya: (574, 3) pozisyon vektörü
⏳ Mars pozisyonları hesaplanıyor...


SpiceSPKINSUFFDATA: 
================================================================================

Toolkit version: CSPICE_N0067

SPICE(SPKINSUFFDATA) --

Insufficient ephemeris data has been loaded to compute the position of 499 (MARS) relative to 10 (SUN) at the ephemeris epoch 2025 JAN 01 00:01:09.183.

spkpos_c --> SPKPOS --> SPKEZP --> SPKGPS

================================================================================

## 7. Apophis'in Dünya'ya Mesafesini Hesapla

In [ ]:
# ─── Apophis'in Dünya'ya mesafesi (AU) ──────────────────────────────────────
apophis_to_earth = np.linalg.norm(apophis_positions - earth_positions, axis=1)

print('📊 Apophis — Dünya Mesafe İstatistikleri (AU):')
print(f'   Min : {apophis_to_earth.min():.4f} AU  ({apophis_to_earth.min()*1.496e8:.0f} km)')
print(f'   Max : {apophis_to_earth.max():.4f} AU  ({apophis_to_earth.max()*1.496e8:.0f} km)')
print(f'   Ort : {apophis_to_earth.mean():.4f} AU')
print(f'\n   PHA kriteri: MOID ≤ 0.05 AU (Apophis bu kriterin altında!)')

# ─── 2029 Yakın geçişini bul ─────────────────────────────────────────────────
# Apophis'in 2029 Nisan'da yakın geçişi biliniyor
mask_2029 = np.array([d.startswith('2029') for d in dates])
if mask_2029.sum() > 0:
    idx_2029_min = np.argmin(apophis_to_earth[mask_2029])
    # 2029 içindeki yerel minimum
    dates_2029 = np.array(dates)[mask_2029]
    dist_2029  = apophis_to_earth[mask_2029]

    closest_date_2029 = dates_2029[idx_2029_min]
    closest_dist_2029 = dist_2029[idx_2029_min]

    print(f'\n🚨 2029 En Yakın Geçiş:')
    print(f'   Tarih    : {closest_date_2029}')
    print(f'   Mesafe   : {closest_dist_2029:.5f} AU  ({closest_dist_2029*1.496e8:.0f} km)')
    print(f'   Referans : Ay Yörüngesi ≈ 0.00257 AU (384,400 km)')

    # 2036 yakın geçişi (bazı tahminlere göre önemli)
    mask_2036 = np.array([d.startswith('2035') or d.startswith('2036') for d in dates])
    if mask_2036.sum() > 0:
        idx_2036_min = np.argmin(apophis_to_earth[mask_2036])
        dates_2036 = np.array(dates)[mask_2036]
        dist_2036  = apophis_to_earth[mask_2036]
        print(f'\n🔭 2035–2036 En Yakın Geçiş:')
        print(f'   Tarih    : {dates_2036[idx_2036_min]}')
        print(f'   Mesafe   : {dist_2036[idx_2036_min]:.5f} AU  ({dist_2036[idx_2036_min]*1.496e8:.0f} km)')

NameError: name 'apophis_positions' is not defined

## 8. Apophis'in Dünya'ya Mesafesi — Zaman Grafiği

In [ ]:
import datetime as dt

fig, ax = plt.subplots(figsize=(14, 6))

# Tarih listesini datetime objesine çevir
date_objects = [dt.datetime.strptime(d, '%Y-%m-%d') for d in dates]

# Ana eğri
ax.plot(date_objects, apophis_to_earth, color='#4ECDC4', linewidth=2, label='Apophis–Dünya Mesafesi')
ax.fill_between(date_objects, apophis_to_earth, alpha=0.15, color='#4ECDC4')

# PHA eşik çizgisi (0.05 AU)
ax.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5, alpha=0.8,
           label='PHA Eşiği (0.05 AU)')

# Ay yörünge mesafesi referansı
ax.axhline(y=0.00257, color='#FFEAA7', linestyle=':', linewidth=1.5, alpha=0.8,
           label='Ay Yörüngesi (0.00257 AU)')

# 2029 yakın geçişini vurgula
idx_global_min_2029 = np.where(mask_2029)[0][np.argmin(apophis_to_earth[mask_2029])]
ax.annotate(
    f'2029 Yakın Geçiş\n{closest_dist_2029:.4f} AU\n({closest_dist_2029*1.496e8:,.0f} km)',
    xy=(date_objects[idx_global_min_2029], apophis_to_earth[idx_global_min_2029]),
    xytext=(dt.datetime(2028, 1, 1), apophis_to_earth[idx_global_min_2029] + 0.15),
    fontsize=10, color='#FF6B6B', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#FF6B6B', lw=2),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='#FF6B6B')
)

# Yıl dikey çizgileri
for year in range(2025, 2037):
    ax.axvline(x=dt.datetime(year, 1, 1), color='gray', alpha=0.2, linewidth=0.8)

ax.set_xlabel('Tarih', fontsize=12)
ax.set_ylabel('Apophis–Dünya Mesafesi (AU)', fontsize=12)
ax.set_title('🚨 Apophis (99942) — Dünya\'ya Mesafe (2025–2035)\n'
             'Yakın geçiş tarihleri ve PHA eşiği', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('apophis_distance_timeline.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()
print('💾 apophis_distance_timeline.png kaydedildi')

## 9. 2D Yörünge Haritası — Güneş Sistemi'ne Genel Bakış

In [ ]:
# ─── 2D görünüm: Ekliptik düzlem (x-y) ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 11))
ax.set_facecolor('#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')

# Güneş
sun = plt.Circle((0, 0), 0.04, color='#FFD93D', zorder=5, label='Güneş')
ax.add_patch(sun)

# Dünya yörüngesi
ax.plot(earth_positions[:, 0], earth_positions[:, 1],
        color='#4ECDC4', linewidth=1.5, alpha=0.7, label='Dünya Yörüngesi')
# Dünya anlık konumu (en son tarih)
ax.scatter(earth_positions[-1, 0], earth_positions[-1, 1],
           color='#4ECDC4', s=80, zorder=6, marker='o')

# Mars yörüngesi
ax.plot(mars_positions[:, 0], mars_positions[:, 1],
        color='#FF6B6B', linewidth=1.5, alpha=0.5, label='Mars Yörüngesi')
ax.scatter(mars_positions[-1, 0], mars_positions[-1, 1],
           color='#FF6B6B', s=80, zorder=6, marker='o')

# Apophis yörüngesi
ax.plot(apophis_positions[:, 0], apophis_positions[:, 1],
        color='#A8E6CF', linewidth=2, alpha=0.9, label='Apophis Yörüngesi')

# 2029 yakın geçiş noktasını vurgula
if mask_2029.sum() > 0:
    ax.scatter(
        apophis_positions[idx_global_min_2029, 0],
        apophis_positions[idx_global_min_2029, 1],
        color='#FFEAA7', s=200, zorder=7, marker='*',
        label=f'2029 Yakın Geçiş ({closest_date_2029})'
    )
    ax.annotate('2029\nYakın Geçiş',
                xy=(apophis_positions[idx_global_min_2029, 0],
                    apophis_positions[idx_global_min_2029, 1]),
                xytext=(apophis_positions[idx_global_min_2029, 0] + 0.15,
                        apophis_positions[idx_global_min_2029, 1] + 0.15),
                color='#FFEAA7', fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#FFEAA7'))

# 1 AU referans çemberi
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), color='gray', alpha=0.2, linewidth=0.8,
        linestyle=':', label='1 AU referansı')

# Eksen etiketleri
ax.set_xlabel('X (AU) — Ekliptik Koordinat', fontsize=11)
ax.set_ylabel('Y (AU) — Ekliptik Koordinat', fontsize=11)
ax.set_title('🌌 İç Güneş Sistemi — Ekliptik Düzlem Görünümü (2025–2035)\n'
             'Apophis, Dünya ve Mars yörüngeleri', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, facecolor='#1a1a2e', edgecolor='gray')
ax.set_aspect('equal')
ax.tick_params(colors='white')
ax.grid(alpha=0.15)

lim = 1.8
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

plt.tight_layout()
plt.savefig('solar_system_2d.png', dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()
print('💾 solar_system_2d.png kaydedildi')

## 10. İnteraktif 3D Yörünge Görselleştirmesi (Plotly)

In [ ]:
fig_3d = go.Figure()

# ─── Güneş ────────────────────────────────────────────────────────────────────
fig_3d.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=12, color='#FFD93D', symbol='circle'),
    name='Güneş',
    hovertext='Güneş (Koordinat Merkezi)'
))

# ─── Dünya yörüngesi ──────────────────────────────────────────────────────────
fig_3d.add_trace(go.Scatter3d(
    x=earth_positions[:, 0],
    y=earth_positions[:, 1],
    z=earth_positions[:, 2],
    mode='lines',
    line=dict(color='#4ECDC4', width=3),
    name='Dünya Yörüngesi',
    text=dates,
    hovertemplate='%{text}<br>X:%{x:.3f} Y:%{y:.3f} Z:%{z:.3f} AU<extra>Dünya</extra>'
))

# ─── Mars yörüngesi ───────────────────────────────────────────────────────────
fig_3d.add_trace(go.Scatter3d(
    x=mars_positions[:, 0],
    y=mars_positions[:, 1],
    z=mars_positions[:, 2],
    mode='lines',
    line=dict(color='#FF6B6B', width=2, dash='dash'),
    name='Mars Yörüngesi',
    opacity=0.5
))

# ─── Apophis yörüngesi ────────────────────────────────────────────────────────
fig_3d.add_trace(go.Scatter3d(
    x=apophis_positions[:, 0],
    y=apophis_positions[:, 1],
    z=apophis_positions[:, 2],
    mode='lines+markers',
    line=dict(color='#A8E6CF', width=4),
    marker=dict(
        size=3,
        color=apophis_to_earth,         # Dünya'ya mesafeye göre renk
        colorscale='RdYlGn_r',          # Kırmızı=yakın, yeşil=uzak
        colorbar=dict(title='Dünya\'a<br>Mesafe (AU)', thickness=15, x=1.0),
        showscale=True,
        cmin=0,
        cmax=apophis_to_earth.max()
    ),
    name='Apophis (99942)',
    text=[f'{d} | {dist:.4f} AU Dünya\'a'
          for d, dist in zip(dates, apophis_to_earth)],
    hovertemplate='%{text}<extra>Apophis</extra>'
))

# ─── 2029 yakın geçiş noktası ────────────────────────────────────────────────
if mask_2029.sum() > 0:
    fig_3d.add_trace(go.Scatter3d(
        x=[apophis_positions[idx_global_min_2029, 0]],
        y=[apophis_positions[idx_global_min_2029, 1]],
        z=[apophis_positions[idx_global_min_2029, 2]],
        mode='markers+text',
        marker=dict(size=14, color='#FFEAA7', symbol='diamond',
                    line=dict(color='white', width=2)),
        text=['2029 Yakın Geçiş'],
        textposition='top center',
        name=f'2029 Yakın Geçiş ({closest_date_2029})',
        hovertext=f'{closest_date_2029}: {closest_dist_2029:.5f} AU ({closest_dist_2029*1.496e8:,.0f} km)'
    ))

# ─── Layout ───────────────────────────────────────────────────────────────────
fig_3d.update_layout(
    title=dict(
        text='🚀 Apophis (99942) — 3D Yörünge Görselleştirmesi (2025–2035)<br>'
             '<sub>Noktaların rengi Dünya\'a olan mesafeyi gösteriyor (kırmızı=yakın)</sub>',
        x=0.5,
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='X (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        yaxis=dict(title='Y (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        zaxis=dict(title='Z (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        bgcolor='#0a0a1a',
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0.5)'),
    width=900,
    height=700,
)

fig_3d.write_html('apophis_3d_orbit.html')
fig_3d.show()
print('💾 apophis_3d_orbit.html kaydedildi (interaktif, tarayıcıda açılabilir)')

## 11. Apophis Yörünge Animasyonu (Plotly)

In [ ]:
# Animasyon: Her frame bir zaman dilimini gösterir
# Performans için her 4. noktayı al
step = 4
anim_dates = dates[::step]
anim_earth  = earth_positions[::step]
anim_apophis = apophis_positions[::step]
anim_distances = apophis_to_earth[::step]
n_frames = len(anim_dates)

print(f'🎬 Animasyon frame sayısı: {n_frames} ({step} adımda bir)')

# ─── Frame verileri ───────────────────────────────────────────────────────────
frames = []
for i in range(n_frames):
    frame_data = [
        # Apophis izi (0'dan i'ye kadar)
        go.Scatter3d(
            x=anim_apophis[:i+1, 0],
            y=anim_apophis[:i+1, 1],
            z=anim_apophis[:i+1, 2],
            mode='lines+markers',
            line=dict(color='#A8E6CF', width=3),
            marker=dict(size=[2]*(i) + [8], color='#A8E6CF'),
        ),
        # Dünya izi
        go.Scatter3d(
            x=anim_earth[:i+1, 0],
            y=anim_earth[:i+1, 1],
            z=anim_earth[:i+1, 2],
            mode='lines+markers',
            line=dict(color='#4ECDC4', width=2),
            marker=dict(size=[2]*(i) + [8], color='#4ECDC4'),
        ),
    ]
    frames.append(go.Frame(data=frame_data, name=anim_dates[i]))

# ─── Başlangıç şekli (ilk frame) ─────────────────────────────────────────────
fig_anim = go.Figure(
    data=[
        go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                     marker=dict(size=10, color='#FFD93D'), name='Güneş'),
        go.Scatter3d(x=anim_apophis[:1, 0], y=anim_apophis[:1, 1], z=anim_apophis[:1, 2],
                     mode='markers', marker=dict(size=8, color='#A8E6CF'), name='Apophis'),
        go.Scatter3d(x=anim_earth[:1, 0], y=anim_earth[:1, 1], z=anim_earth[:1, 2],
                     mode='markers', marker=dict(size=8, color='#4ECDC4'), name='Dünya'),
    ],
    frames=frames
)

fig_anim.update_layout(
    title=dict(text='🎬 Apophis Animasyonu — 2025–2035', x=0.5, font=dict(size=15)),
    scene=dict(
        xaxis=dict(title='X (AU)', range=[-2, 2], backgroundcolor='#0a0a1a'),
        yaxis=dict(title='Y (AU)', range=[-2, 2], backgroundcolor='#0a0a1a'),
        zaxis=dict(title='Z (AU)', range=[-0.5, 0.5], backgroundcolor='#0a0a1a'),
        bgcolor='#0a0a1a',
    ),
    paper_bgcolor='#1a1a2e',
    font=dict(color='white'),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=1.05, x=0.5,
        buttons=[
            dict(label='▶ Oynat',
                 method='animate',
                 args=[None, dict(frame=dict(duration=80, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Durdur',
                 method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')])
        ]
    )],
    sliders=[dict(
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name[::30]) for f in frames],
        active=0, y=0, len=1.0, x=0, currentvalue=dict(prefix='Tarih: ', font=dict(size=12))
    )],
    width=900, height=700,
)

fig_anim.write_html('apophis_animation.html')
fig_anim.show()
print('💾 apophis_animation.html kaydedildi')

## 12. Tehlike Penceresi Analizi — Yakın Geçiş Yakınlaştırması

In [ ]:
# 2028–2030 dönemini yakınlaştır — en kritik periyot
mask_critical = np.array([d >= '2028-01-01' and d <= '2030-12-31' for d in dates])
dates_critical  = np.array(date_objects)[mask_critical]
dist_critical   = apophis_to_earth[mask_critical]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ─── Panel 1: Tüm dönem ───────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(date_objects, apophis_to_earth, color='#4ECDC4', linewidth=1.5)
ax1.fill_between(date_objects, 0, apophis_to_earth,
                 where=apophis_to_earth < 0.05,
                 alpha=0.3, color='#FF6B6B', label='PHA tehlike bölgesi')
ax1.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5, label='PHA Eşiği (0.05 AU)')
ax1.axvline(x=dt.datetime(2029, 4, 13), color='#FFEAA7', linestyle=':', linewidth=1.5,
            label='Tahmin: 13 Nisan 2029')

ax1.set_title('Apophis–Dünya Mesafesi — Tam Dönem (2025–2035)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mesafe (AU)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.2)
ax1.set_ylim(0)

# Zoom alanını kutucukla göster
from matplotlib.patches import Rectangle
ax1.add_patch(Rectangle(
    (dt.datetime(2028, 1, 1), 0), dt.timedelta(days=3*365), 0.6,
    linewidth=2, edgecolor='#FFEAA7', facecolor='none', linestyle='--', alpha=0.7
))
ax1.text(dt.datetime(2028, 2, 1), 0.55, 'Yakınlaştırılan bölge →',
         color='#FFEAA7', fontsize=9)

# ─── Panel 2: Yakınlaştırma ───────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(dates_critical, dist_critical, color='#FFEAA7', linewidth=2.5)
ax2.fill_between(dates_critical, 0, dist_critical,
                 where=dist_critical < 0.05,
                 alpha=0.4, color='#FF6B6B')
ax2.fill_between(dates_critical, 0, dist_critical,
                 where=dist_critical >= 0.05,
                 alpha=0.15, color='#4ECDC4')

# Eşik çizgileri
ax2.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5,
            label='PHA Eşiği (0.05 AU)')
ax2.axhline(y=0.00257, color='#96CEB4', linestyle=':', linewidth=1.5,
            label='Ay Yörüngesi (0.00257 AU)')

# Minimum noktayı işaretle
if mask_critical.sum() > 0:
    idx_min = np.argmin(dist_critical)
    ax2.scatter(dates_critical[idx_min], dist_critical[idx_min],
                color='#FF4444', s=200, zorder=5, label=f'En yakın: {dist_critical[idx_min]:.4f} AU')
    ax2.annotate(
        f'Min: {dist_critical[idx_min]:.5f} AU
({dist_critical[idx_min]*1.496e8:,.0f} km)',
        xy=(dates_critical[idx_min], dist_critical[idx_min]),
        xytext=(dates_critical[idx_min], dist_critical[idx_min] + 0.04),
        fontsize=9, color='#FF4444', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#FF4444')
    )

ax2.set_title('🔍 2028–2030 Yakınlaştırma — Apophis Tehlike Penceresi', fontsize=12, fontweight='bold')
ax2.set_xlabel('Tarih')
ax2.set_ylabel('Mesafe (AU)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)
ax2.set_ylim(0)

plt.suptitle('🚨 Apophis Tehlike Analizi', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('apophis_danger_window.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()
print('💾 apophis_danger_window.png kaydedildi')

## 13. Önemli Tarihler ve Veriler — Özet Tablosu

In [ ]:
# ─── Özet istatistikler ──────────────────────────────────────────────────────
# PHA eşiği altında geçen süreyi hesapla
pha_below = apophis_to_earth < 0.05
days_below = pha_below.sum() * step_days  # yaklaşık gün sayısı

# Yıllık minimum mesafe
print('📊 YILLIK MİNİMUM MESAFE TABLOSU')
print('='*55)
print(f'{'Yıl':<8} {'Min Mesafe (AU)':<20} {'Min Mesafe (km)':<20} {'Tarih':<15}')
print('-'*55)

annual_summary = []
for year in range(2025, 2036):
    mask_year = np.array([d.startswith(str(year)) for d in dates])
    if mask_year.sum() > 0:
        idx_min = np.argmin(apophis_to_earth[mask_year])
        dates_year = np.array(dates)[mask_year]
        dist_year  = apophis_to_earth[mask_year]
        min_dist   = dist_year[idx_min]
        min_date   = dates_year[idx_min]
        is_pha = '🚨' if min_dist < 0.05 else '  '
        print(f'{year:<8} {min_dist:<20.5f} {min_dist*1.496e8:<20,.0f} {min_date:<15} {is_pha}')
        annual_summary.append({'Yıl': year, 'Min AU': min_dist,
                                'Min km': min_dist*1.496e8, 'Tarih': min_date})

print('='*55)
print(f'🚨 = PHA eşiği altında (< 0.05 AU)')
print(f'\n📅 PHA eşiği altında geçen yaklaşık süre: ~{days_below} gün')

# Pandas tablosuna da çevir
summary_df = pd.DataFrame(annual_summary)
print('\n')
print(summary_df.to_string(index=False))

## 14. Drive'a Kaydet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

DRIVE_PATH = '/content/drive/MyDrive/nasa_asteroid/'
os.makedirs(DRIVE_PATH, exist_ok=True)

# Görselleri Drive'a kopyala
outputs = [
    'apophis_distance_timeline.png',
    'solar_system_2d.png',
    'apophis_3d_orbit.html',
    'apophis_animation.html',
    'apophis_danger_window.png',
]

print('💾 Dosyalar Drive\'a kaydediliyor...')
for fname in outputs:
    if os.path.exists(fname):
        shutil.copy(fname, DRIVE_PATH + fname)
        print(f'   ✅ {fname}')
    else:
        print(f'   ⚠️  {fname} bulunamadı')

# Pozisyon verilerini numpy olarak kaydet
np.save('apophis_positions.npy', apophis_positions)
np.save('earth_positions.npy', earth_positions)
np.save('mars_positions.npy', mars_positions)
np.save('distances_au.npy', apophis_to_earth)

for fname in ['apophis_positions.npy', 'earth_positions.npy',
              'mars_positions.npy', 'distances_au.npy']:
    shutil.copy(fname, DRIVE_PATH + fname)
    print(f'   ✅ {fname}')

print(f'\n🚀 Notebook 03 tamamlandı!')
print(f'   Sıradaki adım: 04_smote_challenge.ipynb')

## 📋 Bu Notebook'tan Çıkarımlar

### SPICE ile Yapılanlar
| Görev | Araç | Çıktı |
|-------|------|-------|
| Gezegen pozisyonu | `spice.spkpos()` | x, y, z koordinatları (AU) |
| Zaman dönüşümü | `spice.str2et()` | Ephemeris Time |
| Kernel yükleme | `spice.furnsh()` | de440s.bsp, naif0012.tls |
| Apophis yörüngesi | SPK kernel / Kepler | 11 yıllık pozisyon dizisi |

### Önemli Bulgular
- **2029 Yakın Geçiş**: Apophis, Dünya'ya tarihsel olarak yakın geçecek
- **PHA Kriteri**: MOID ≤ 0.05 AU — Apophis bu eşiğin altına iniyor
- **Ay Yörüngesi Referansı**: 0.00257 AU ≈ 384,400 km
- **Görsel Doğrulama**: ML modelimizin tehlikeli dediği asteroidin neden tehlikeli olduğunu gösterdik

### Raporun İçin Not Et
- `apophis_3d_orbit.html` interaktif — sunumda tarayıcıda göster
- 2D ekliptik haritası: Apophis'in Dünya yörüngesiyle kesiştiğini gösteriyor
- Kepler fallback vs SPICE farkını tartış (kernel yoksa nasıl devam edilir?)
- SPICE'ın NASA görevlerinde kullanımından bahset (bağlam için)

### ⏭️ Sıradaki Notebook
`04_smote_challenge.ipynb` → SMOTE ile sınıf dengesizliğini çözme